# Design exercise D5 · Help-centre search — Part B

Contapiù's help centre answers customers with articles. Most customers read
only the first result or two, so for each query the article that answers it
should be **as near the top as possible**. Your job: for each of 2,000 July
queries, hand in the ten best article ids, best first.

You have about 15 minutes. The cells below load the data, define the metric,
set up a validation month, score a baseline and write the file. Make **your**
design choices in the cell marked *Your model*, compare them on the
validation month, then hand in.

**Scoring.** Your ten articles for each query are compared with the article
the support team judged to answer it, which only the grader has. The mean
reciprocal rank over the first ten (1 if the answer is first, 1/2 if second,
..., 0 if it is not in your ten) gives the points: 0 at the level of a random
ranking (0.01), 10 at the reference design's 0.83 or above, linear in between.

> **One attempt.** Only the **first** file you turn in is scored. If you
> unsubmit and turn in another file, it is not scored: you keep the score of
> the first. Choose your model with the notebook's own validation, check the file with the last
> cell, and only then turn it in.

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import normalize

BASE = "https://raw.githubusercontent.com/fabsilvestri/AIML-Course/main/design/d05/"
articles = pd.read_csv(BASE + "articles.csv")
log = pd.read_csv(BASE + "train.csv")        # January to June: query, click, ticket
new = pd.read_csv(BASE + "test.csv")         # the 2,000 July queries to rank for

assert articles["article_id"].is_unique and len(articles) == 274
assert log.shape == (9000, 5) and new.shape == (2000, 3)
assert log["article_id"].isin(articles["article_id"]).all()
ids = articles["article_id"].to_numpy()
texts = (articles["title"] + " " + articles["body"]).tolist()
print(f"{len(articles)} articles; {len(log):,} logged queries, "
      f"{log['ticket'].mean():.1%} followed by a support ticket; "
      f"{len(new):,} July queries")
log.sample(5, random_state=0)

The metric is the one the brief fixes: the reciprocal rank of the answering
article within your first ten, averaged over the queries.

The validation fold is a month, as in deployment: a model is fitted on the log
up to the end of May and judged on June's queries, then refitted on January to
June for July. A click followed by a support ticket did not answer its query,
so June's judged answers are the clicks with no ticket after them.

A model here is a function `score(rows, queries)`: given the log rows it may
learn from, it returns a matrix of scores, one row per query and one column
per article (in the order of `ids`). Higher means a better answer.

In [ ]:
def top10(S):
    """The ten highest-scored article ids for each row of S."""
    return [list(ids[row]) for row in np.argsort(-S, axis=1, kind="stable")[:, :10]]

def mrr_at_10(tops, answers):
    """Mean over queries of 1/rank of the answer within the first ten, else 0."""
    rr = [next((1 / (r + 1) for r, a in enumerate(t[:10]) if a == ans), 0.0)
          for t, ans in zip(tops, answers)]
    return float(np.mean(rr))

fit_rows = log[log["date"] < "2026-06-01"]
june = log[(log["date"] >= "2026-06-01") & (log["ticket"] == 0)]

def validate(score):
    """Fit on January to May, rank June's queries, return MRR@10."""
    S = score(fit_rows, june["query"].tolist())
    assert S.shape == (len(june), len(ids))
    return mrr_at_10(top10(S), june["article_id"].tolist())

print(f"fit on {len(fit_rows):,} queries; validate on {len(june):,} June queries")

A baseline, to beat: BM25 over each article's title and body, with k1 = 1.2
and b = 0.75, written out as in Lecture 19. It learns nothing from the log.

In [ ]:
TOKENS = r"[a-z0-9]+"

def bm25(docs, k1=1.2, b=0.75):
    """BM25 weight of every term in every document, and the vectoriser."""
    cv = CountVectorizer(token_pattern=TOKENS)
    tf = cv.fit_transform(docs).astype(float).tocoo()
    N = tf.shape[0]
    n_t = np.bincount(tf.col, minlength=tf.shape[1])
    idf = np.log(1 + (N - n_t + 0.5) / (n_t + 0.5))
    dl = np.bincount(tf.row, weights=tf.data, minlength=N)
    norm = k1 * (1 - b + b * dl / dl.mean())
    w = idf[tf.col] * tf.data * (k1 + 1) / (tf.data + norm[tf.row])
    return sparse.csr_matrix((w, (tf.row, tf.col)), shape=tf.shape), cv

W, cv = bm25(texts)

def score_bm25(rows, queries):
    return (cv.transform(queries) @ W.T).toarray()

print(f"BM25: {validate(score_bm25):.3f}")

## Your model

Build your design here as a function `score(rows, queries)`, and compare it
with the baseline on the validation month. Every choice — which log rows to
learn from, how queries and articles are represented, which model — is yours,
and is judged by `validate`.

In [ ]:
def score(rows, queries):
    return score_bm25(rows, queries)          # replace with your design

print(f"your model: {validate(score):.3f}")

## Hand in

Fit on the whole log, January to June, rank the July queries, and check the
file before you write it.

In [ ]:
S = score(log, new["query"].tolist())
tops = top10(S)

out = pd.DataFrame({"query_id": new["query_id"],
                    "articles": [" ".join(t) for t in tops]})
assert len(out) == 2000 and out["query_id"].is_unique
assert set(out["query_id"]) == set(new["query_id"])
assert out["articles"].notna().all()
for t in tops:
    assert len(t) == 10 and len(set(t)) == 10, "ten distinct articles per query"
    assert set(t) <= set(ids), "unknown article id"
out.to_csv("rankings.csv", index=False)
print("wrote rankings.csv:", len(out), "rows")
out.head()

In Colab, download it (the folder icon on the left, or run the next cell),
then in Classroom open the assignment, **Add → File**, choose
`rankings.csv`, and **Turn in**. The grader marks it within the hour.
**Only this first file counts**: a second one is not scored.

In [ ]:
try:
    from google.colab import files
    files.download("rankings.csv")
except ImportError:
    print("Not in Colab: rankings.csv is in the working directory.")